In [ ]:
# 1 重采样tas era5-daily到周的1.5度
import os
import xarray as xr
import numpy as np
from tqdm import tqdm

# 设置路径
input_dir = '/mnt/g/数据/ECMWF数据/ERA5-daily/ERA5-daily-single level-2m_Temperature'
output_base_dir = '/mnt/g/次季节模型/数据/obs-era5/tas'

# 目标插值的经纬度
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# 获取所有 NetCDF 文件
nc_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.nc')])

for file in tqdm(nc_files, desc='📦 正在处理 ERA5 文件'):
    year = file[-7:-3]
    file_path = os.path.join(input_dir, file)

    # 打开数据集并提取 t2m
    ds = xr.open_dataset(file_path)
    t2m = ds['t2m']  # (valid_time, latitude, longitude)

    # 转换为 °C（单位为 K）
    #t2m_c = t2m - 273.15
    t2m_c = t2m
    t2m_c.name = 'tas'
    t2m_c.attrs['units'] = 'K'
    t2m_c.attrs['long_name'] = '2 metre temperature'
    t2m_c.attrs['standard_name'] = 'air_temperature'

    # 使用 valid_time 重采样为每周均值，以周一为起始
    t2m_weekly = t2m_c.resample(valid_time='1W-MON').mean()

    # 插值到目标网格
    t2m_interp = t2m_weekly.interp(latitude=target_lat, longitude=target_lon, method='linear')

    # 遍历每一周，保存为独立文件
    for i in range(t2m_interp.valid_time.size):
        da = t2m_interp.isel(valid_time=i)
        date_str = np.datetime_as_string(da.valid_time.values, unit='D').replace('-', '')
        output_path = os.path.join(output_base_dir, year, f'obs-era5-{date_str}-tas.nc')
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # 构造保存的数据集
        da_ds = xr.Dataset({da.name: da})

        # 添加单一时间坐标
        da_ds = da_ds.expand_dims('time')
        da_ds = da_ds.assign_coords(time=[da.valid_time.values])
        da_ds['time'].attrs = {
            'standard_name': 'time',
            'long_name': 'time',
            'bounds': 'time_bnds',
            'axis': 'T'
        }

        # 添加 variable 坐标
        da_ds['variable'] = xr.DataArray(['tas'], dims='variable')
        da_ds['variable'].attrs = {
            'long_name': 'variable name',
            'standard_name': 'variable',
        }

        # 保存为 NetCDF 文件
        da_ds.to_netcdf(output_path)

In [ ]:
# 2 计算每周高度层数据，1941、1949、2024无法处理
import os
import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm

# 输入输出路径
root_dir = "/mnt/g/数据/ECMWF数据/ERA5-daily"
output_dir = "/mnt/g/次季节模型/数据/obs-era5/hi_gh"
os.makedirs(output_dir, exist_ok=True)

# 文件路径模板
file_template = os.path.join(root_dir, "ERA5-daily-{level}hpa-Geopotential", "ERA5-daily-{level}hpa-Geopotential-{year}.nc")

# 重采样目标网格
target_lat = np.arange(90, -91, -1.5)
target_lon = np.arange(0, 360, 1.5)

# 年份范围
years = range(1940, 2025)

for year in tqdm(years, desc="📦 正在处理 ERA5 位势高度文件"):
    try:
        ds200 = xr.open_dataset(file_template.format(level=200, year=year))
        ds300 = xr.open_dataset(file_template.format(level=300, year=year))
        ds500 = xr.open_dataset(file_template.format(level=500, year=year))

        z200 = ds200['z'].squeeze() / 9.8
        z300 = ds300['z'].squeeze() / 9.8
        z500 = ds500['z'].squeeze() / 9.8

        z200_interp = z200.interp(latitude=target_lat, longitude=target_lon, method="linear")
        z300_interp = z300.interp(latitude=target_lat, longitude=target_lon, method="linear")
        z500_interp = z500.interp(latitude=target_lat, longitude=target_lon, method="linear")

        z200_filled = z200_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')
        z300_filled = z300_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')
        z500_filled = z500_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')

        z200_z300 = z200_filled - z300_filled
        z200_z500 = z200_filled - z500_filled

        time_dim = 'valid_time' if 'valid_time' in z200_filled.dims else 'time'

        z200_weekly = z200_filled.resample({time_dim: '1W-MON'}).mean()
        z200_z300_weekly = z200_z300.resample({time_dim: '1W-MON'}).mean()
        z200_z500_weekly = z200_z500.resample({time_dim: '1W-MON'}).mean()

        # 新增：创建年份子目录
        year_output_dir = os.path.join(output_dir, str(year))
        os.makedirs(year_output_dir, exist_ok=True)

        for i in range(len(z200_weekly[time_dim])):
            week_date = pd.to_datetime(str(z200_weekly[time_dim][i].values)).strftime("%Y%m%d")
            ds_week = xr.Dataset({
                "z200_weekly": z200_weekly.isel({time_dim: i}),
                "z200_z300_weekly": z200_z300_weekly.isel({time_dim: i}),
                "z200_z500_weekly": z200_z500_weekly.isel({time_dim: i}),
            })
            filename = f"obs-era5-{week_date}-gh.nc"
            ds_week.to_netcdf(os.path.join(year_output_dir, filename))

        print(f"✅ 已处理 {year} 年，共 {len(z200_weekly[time_dim])} 周，样例变量：")
        print(ds_week)
        print(f"变量 z200_weekly 最大值: {ds_week['z200_weekly'].max().item():.3f}, 最小值: {ds_week['z200_weekly'].min().item():.3f}")
        print(f"变量 z200_z300_weekly 最大值: {ds_week['z200_z300_weekly'].max().item():.3f}, 最小值: {ds_week['z200_z300_weekly'].min().item():.3f}")
        print(f"变量 z200_z500_weekly 最大值: {ds_week['z200_z500_weekly'].max().item():.3f}, 最小值: {ds_week['z200_z500_weekly'].min().item():.3f}")


    except Exception as e:
        print(f"❌ 无法处理 {year} 年的数据: {e}")


In [ ]:
# 2 计算每周高度层数据2024
import os
import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm

# 输入输出路径
root_dir = "/mnt/g/数据/ECMWF数据/ERA5-daily"
output_dir = "/mnt/g/次季节模型/数据/obs-era5/hi_gh"
os.makedirs(output_dir, exist_ok=True)

# 文件路径模板
file_template = os.path.join(root_dir, "ERA5-daily-{level}hpa-Geopotential", "ERA5-daily-{level}hpa-Geopotential-{year}.nc")

# 重采样目标网格
target_lat = np.arange(90, -91, -1.5)
target_lon = np.arange(0, 360, 1.5)

# 指定年份为2024
year = 2024
print(f"📦 正在处理 ERA5 位势高度文件：{year}")

try:
    ds200 = xr.open_dataset(file_template.format(level=200, year=year))
    ds300 = xr.open_dataset(file_template.format(level=300, year=year))
    ds500 = xr.open_dataset(file_template.format(level=500, year=year))

    z200 = ds200['z'].squeeze() / 9.8
    z300 = ds300['z'].squeeze() / 9.8
    z500 = ds500['z'].squeeze() / 9.8

    z200_interp = z200.interp(latitude=target_lat, longitude=target_lon, method="linear")
    z300_interp = z300.interp(latitude=target_lat, longitude=target_lon, method="linear")
    z500_interp = z500.interp(latitude=target_lat, longitude=target_lon, method="linear")

    z200_filled = z200_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')
    z300_filled = z300_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')
    z500_filled = z500_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')

    z200_z300 = z200_filled - z300_filled
    z200_z500 = z200_filled - z500_filled

    time_dim = 'valid_time' if 'valid_time' in z200_filled.dims else 'time'

    z200_weekly = z200_filled.resample({time_dim: '1W-MON'}).mean()
    z200_z300_weekly = z200_z300.resample({time_dim: '1W-MON'}).mean()
    z200_z500_weekly = z200_z500.resample({time_dim: '1W-MON'}).mean()

    # 创建年份子目录
    year_output_dir = os.path.join(output_dir, str(year))
    os.makedirs(year_output_dir, exist_ok=True)

    for i in range(len(z200_weekly[time_dim])):
        week_date = pd.to_datetime(str(z200_weekly[time_dim][i].values)).strftime("%Y%m%d")
        ds_week = xr.Dataset({
            "z200_weekly": z200_weekly.isel({time_dim: i}),
            "z200_z300_weekly": z200_z300_weekly.isel({time_dim: i}),
            "z200_z500_weekly": z200_z500_weekly.isel({time_dim: i}),
        })
        filename = f"obs-era5-{week_date}-gh.nc"
        ds_week.to_netcdf(os.path.join(year_output_dir, filename))

    print(f"✅ 已处理 {year} 年，共 {len(z200_weekly[time_dim])} 周，样例变量：")
    print(ds_week)
    print(f"变量 z200_weekly 最大值: {ds_week['z200_weekly'].max().item():.3f}, 最小值: {ds_week['z200_weekly'].min().item():.3f}")
    print(f"变量 z200_z300_weekly 最大值: {ds_week['z200_z300_weekly'].max().item():.3f}, 最小值: {ds_week['z200_z300_weekly'].min().item():.3f}")
    print(f"变量 z200_z500_weekly 最大值: {ds_week['z200_z500_weekly'].max().item():.3f}, 最小值: {ds_week['z200_z500_weekly'].min().item():.3f}")

except Exception as e:
    print(f"❌ 无法处理 {year} 年的数据: {e}")


In [ ]:
# 3 合并周数据
import os
import xarray as xr

tas_root = "/mnt/g/次季节模型/数据/obs-era5/tas"
gh_root = "/mnt/g/次季节模型/数据/obs-era5/hi_gh"
output_year_dir = "/mnt/g/次季节模型/数据/obs-era5/tas-hi_gh_year"
os.makedirs(output_year_dir, exist_ok=True)

def process_single_year(year):
    print(f"Processing year {year}...")
    tas_dir = os.path.join(tas_root, str(year))
    gh_dir = os.path.join(gh_root, str(year))

    if not os.path.exists(tas_dir) or not os.path.exists(gh_dir):
        print(f"  ⚠️ Missing directory for year {year}, skipping.")
        return

    tas_files = sorted([f for f in os.listdir(tas_dir) if f.endswith(".nc")])
    gh_files = sorted([f for f in os.listdir(gh_dir) if f.endswith(".nc")])

    tas_map = {f.split("-")[2]: f for f in tas_files}
    gh_map = {f.split("-")[2]: f for f in gh_files}
    common_dates = sorted(set(tas_map.keys()) & set(gh_map.keys()))

    merged_datasets = []

    for date_str in common_dates:
        tas_file = os.path.join(tas_dir, tas_map[date_str])
        gh_file = os.path.join(gh_dir, gh_map[date_str])

        try:
            with xr.open_dataset(tas_file) as ds_tas, xr.open_dataset(gh_file) as ds_gh:
                # 删除hi_gh中的valid_time坐标
                ds_gh = ds_gh.reset_coords('valid_time', drop=True)

                # 取tas的第一个time值（周一）
                time_val = ds_tas['time'].values[0:1]

                # 扩展ds_gh时间维度为time_val，时间坐标为tas的time
                ds_gh_expanded = ds_gh.expand_dims({'time': time_val})

                # 取tas对应时间点的tas数据，扩展time维度
                tas_sel = ds_tas['tas'].isel(time=0).expand_dims('time')

                # 构造新数据集，时间坐标统一用tas的time
                ds_merged = xr.Dataset(
                    data_vars={
                        'tas': tas_sel,
                        'z200_weekly': ds_gh_expanded['z200_weekly'],
                        'z200_z300_weekly': ds_gh_expanded['z200_z300_weekly'],
                        'z200_z500_weekly': ds_gh_expanded['z200_z500_weekly'],
                    },
                    coords={
                        'time': time_val,
                        'latitude': ds_tas['latitude'],
                        'longitude': ds_tas['longitude'],
                    }
                )
                merged_datasets.append(ds_merged)

        except Exception as e:
            print(f"  ❌ Error on {date_str}: {e}")

    if merged_datasets:
        ds_year = xr.concat(merged_datasets, dim='time').sortby('time')

        # 额外确保合并后删除valid_time坐标（如果残留）
        if 'valid_time' in ds_year.coords:
            ds_year = ds_year.reset_coords('valid_time', drop=True)

        output_path = os.path.join(output_year_dir, f"merged_{year}.nc")
        print(f"✅ Saving: {output_path}")
        ds_year.to_netcdf(output_path)
        ds_year.close()
    else:
        print(f"⚠️ No valid data found for year {year}")

# 按年处理并保存
for year in range(1999, 2025):
    process_single_year(year)

print("🎯 每年数据处理完成")

In [ ]:
# 4 将合并好的周数据合并
import os
import xarray as xr

output_year_dir = "/mnt/g/次季节模型/数据/obs-era5/tas-hi_gh_year/"
output_all_file = "/mnt/g/次季节模型/数据/obs-era5/tas-hi_gh_1999-2024_years.nc"

print("📦 Merging all yearly files...")

year_files = sorted([
    os.path.join(output_year_dir, f) 
    for f in os.listdir(output_year_dir) if f.endswith(".nc")
])

ds_all = xr.open_mfdataset(year_files, combine='by_coords')
ds_all = ds_all.sortby('time')
ds_all.to_netcdf(output_all_file)
ds_all.close()

print(f"✅ All years merged and saved to:\n{output_all_file}")


In [ ]:
# 5 日数据提取到1.5度
import os
import xarray as xr
import numpy as np
from tqdm import tqdm

# ========== 路径配置 ==========
input_dir = "/mnt/g/数据/ECMWF数据/ERA5-daily/ERA5-daily-single level-2m_Temperature"
output_base_dir = "/mnt/g/次季节模型/数据/日数据/tas"
os.makedirs(output_base_dir, exist_ok=True)

# ========== 目标插值网格 ==========
new_lats = np.arange(-90, 90.1, 1.5)  # 纬度升序
new_lons = np.arange(0, 360, 1.5)

# ========== 处理目标年份 ==========
target_years = [str(y) for y in range(2024, 2025)]

# ========== 主循环 ==========
for filename in tqdm(sorted(os.listdir(input_dir))):
    if not filename.endswith(".nc"):
        continue

    year = filename.split("-")[-1].split(".")[0]
    if year not in target_years:
        continue

    file_path = os.path.join(input_dir, filename)
    output_file = os.path.join(output_base_dir, f"tas_{year}.nc")
    print(f"📦 正在处理 {file_path}")

    try:
        ds = xr.open_dataset(file_path)

        # 确保时间维度统一命名为 "time"
        if "valid_time" in ds.coords:
            ds = ds.rename({"valid_time": "time"})

        # 确保 "time" 是坐标维度
        if "time" not in ds.dims:
            ds = ds.swap_dims({"time": "time"})

        tas = ds["t2m"]

        # 纬度升序
        if tas.latitude[0] > tas.latitude[-1]:
            tas = tas.sortby("latitude")

        # 插值
        tas_interp = tas.interp(latitude=new_lats, longitude=new_lons, method="linear")

        # 插值后填补 NaN
        tas_filled = tas_interp.interpolate_na(dim="latitude", method="nearest", fill_value="extrapolate")
        tas_filled = tas_filled.interpolate_na(dim="longitude", method="nearest", fill_value="extrapolate")

        # 创建输出数据集
        ds_out = xr.Dataset(
            {"tas": tas_filled},
            coords={
                "time": ds["time"].values,  # 标准化时间坐标
                "latitude": new_lats,
                "longitude": new_lons,
            }
        )

        # 设置属性
        ds_out["tas"].attrs = tas.attrs
        ds_out["tas"].attrs.update({
            "long_name": "2 metre temperature",
            "units": "K"
        })
        ds_out.attrs = {k: v for k, v in ds.attrs.items() if isinstance(v, (str, int, float))}

        # 保存结果
        ds_out.to_netcdf(output_file)
        print(f"✅ 保存成功: {output_file}")

    except Exception as e:
        print(f"❌ 处理失败: {file_path}，错误: {e}")


In [ ]:
# 6 基准日期和7日日数据对应
import os
import xarray as xr
import pandas as pd
from datetime import timedelta

def generate_target_dates(week_monday):
    """
    给定某个周一日期，返回该周和下周对应的7个目标日期：
    周一、周三、周五、周日、下周二、下周四、下周六
    """
    dates = []
    # 本周
    dates.append(week_monday)                     # 周一
    dates.append(week_monday + timedelta(days=2)) # 周三
    dates.append(week_monday + timedelta(days=4)) # 周五
    dates.append(week_monday + timedelta(days=6)) # 周日
    # 下周
    dates.append(week_monday + timedelta(days=8))  # 下周二
    dates.append(week_monday + timedelta(days=10)) # 下周四
    dates.append(week_monday + timedelta(days=12)) # 下周六
    return dates

def load_multi_year_data(year, input_dir):
    """
    读取指定年份和下一年（若存在）数据并合并，
    以满足跨年日期提取需求。
    """
    ds_current = None
    infile_current = os.path.join(input_dir, f"tas_{year}.nc")
    if os.path.exists(infile_current):
        ds_current = xr.open_dataset(infile_current)
    else:
        print(f"{infile_current} 不存在，跳过该年份")
        return None

    # 读取下一年数据（可能不存在）
    infile_next = os.path.join(input_dir, f"tas_{year+1}.nc")
    ds_next = None
    if os.path.exists(infile_next):
        ds_next = xr.open_dataset(infile_next)

    if ds_next is not None:
        # 合并两个数据集，时间维度拼接
        ds_all = xr.concat([ds_current, ds_next], dim='time')
        # 确保时间排序
        ds_all = ds_all.sortby('time')
        return ds_all
    else:
        return ds_current

def process_year(year, input_dir, output_dir):
    """
    处理某一年份的tas数据，支持跨年：
    - 载入当年和下一年数据
    - 以当年第一个周一至倒数第二个周一为基准
    - 提取7个目标日期数据，缺失则跳过该周
    - 保存按年拆分的数据（只保存基准周一属于该年的数据）
    """
    ds = load_multi_year_data(year, input_dir)
    if ds is None:
        return

    tas = ds['tas']
    time_index = pd.to_datetime(tas.time.values)

    # 取当年范围内的所有周一，基于ds中的时间筛选
    # 只取基准周一属于当年的日期作为有效基准
    mondays_all = time_index[time_index.to_series().dt.weekday == 0]
    mondays = mondays_all[mondays_all.to_series().dt.year == year]

    if len(mondays) < 2:
        print(f"{year} 年周一数量不足，跳过")
        return

    start_monday = mondays[0]
    end_monday = mondays[-2]

    base_mondays = pd.date_range(start=start_monday, end=end_monday, freq='7D')

    data_vars = {
        'tas_mon': [],
        'tas_wed': [],
        'tas_fri': [],
        'tas_sun': [],
        'tas_tue_next': [],
        'tas_thu_next': [],
        'tas_sat_next': [],
    }

    times_out = []

    for base_mon in base_mondays:
        target_dates = generate_target_dates(base_mon)

        # 检查所有目标日期是否存在于数据时间坐标中
        if any(date not in time_index for date in target_dates):
            print(f"{year} {base_mon.date()} 周目标日期跨年缺失，跳过")
            continue

        # 逐个日期提取tas数据
        data_vars['tas_mon'].append(tas.sel(time=target_dates[0]))
        data_vars['tas_wed'].append(tas.sel(time=target_dates[1]))
        data_vars['tas_fri'].append(tas.sel(time=target_dates[2]))
        data_vars['tas_sun'].append(tas.sel(time=target_dates[3]))
        data_vars['tas_tue_next'].append(tas.sel(time=target_dates[4]))
        data_vars['tas_thu_next'].append(tas.sel(time=target_dates[5]))
        data_vars['tas_sat_next'].append(tas.sel(time=target_dates[6]))

        times_out.append(base_mon)

    if len(times_out) == 0:
        print(f"{year} 无有效基准周一，未生成任何数据")
        return

    # 合并为新的时间维度
    for key in data_vars:
        data_vars[key] = xr.concat(data_vars[key], dim='time')
    times_out = pd.to_datetime(times_out)

    ds_out = xr.Dataset(
        {
            'tas_mon': (('time', 'latitude', 'longitude'), data_vars['tas_mon'].data),
            'tas_wed': (('time', 'latitude', 'longitude'), data_vars['tas_wed'].data),
            'tas_fri': (('time', 'latitude', 'longitude'), data_vars['tas_fri'].data),
            'tas_sun': (('time', 'latitude', 'longitude'), data_vars['tas_sun'].data),
            'tas_tue_next': (('time', 'latitude', 'longitude'), data_vars['tas_tue_next'].data),
            'tas_thu_next': (('time', 'latitude', 'longitude'), data_vars['tas_thu_next'].data),
            'tas_sat_next': (('time', 'latitude', 'longitude'), data_vars['tas_sat_next'].data),
        },
        coords={
            'time': times_out,
            'latitude': ds.latitude,
            'longitude': ds.longitude,
        }
    )

    ds_out.attrs['description'] = (
        f"从{year}年第一个周一开始，到倒数第二个周一，提取7个目标日期的tas日均数据，"
        "支持跨年，时间坐标为基准周一。变量分别对应：周一、周三、周五、周日、下周二、下周四、下周六。"
    )

    os.makedirs(output_dir, exist_ok=True)
    outfile = os.path.join(output_dir, f"tas_weekly_split_{year}.nc")
    ds_out.to_netcdf(outfile)
    print(f"{year} 年处理完成，保存至 {outfile}")

def main():
    input_dir = "/mnt/g/次季节模型/数据/日数据/tas"
    output_dir = "/mnt/g/次季节模型/数据/日tas"

    for year in range(2000, 2025):
        process_year(year, input_dir, output_dir)

if __name__ == "__main__":
    main()


In [ ]:
# 7 周数据添加基准日期
import xarray as xr
import pandas as pd
import numpy as np
import os

# 输入输出路径
input_path = "/mnt/g/次季节模型/数据/obs-era5/tas-hi_gh_1999-2024_years.nc"
output_root = "/mnt/g/次季节模型/数据/周数据"

# 加载数据集并确保时间唯一
ds = xr.open_dataset(input_path)
ds['time'] = pd.to_datetime(ds['time'].values)
_, unique_idx = np.unique(ds['time'], return_index=True)
ds = ds.isel(time=unique_idx)

# 重命名变量方便后续调用
ds = ds.rename({
    'z200_weekly': 'gh200',
    'z200_z300_weekly': 'gh200_300',
    'z200_z500_weekly': 'gh200_500'
})

# 构造2000年第一个周一到2024年倒数第二个周一的所有周一日期
start_date = pd.to_datetime("2000-01-03")
end_date = pd.to_datetime("2024-12-16")  # 倒数第二个周一
all_mondays = pd.date_range(start=start_date, end=end_date, freq="W-MON")

target_dates = all_mondays

year_check = {}
results_for_year = []
current_year = None

for target_date in target_dates:
    try:
        year = target_date.year

        # 如果年份变化且非第一次循环，保存上一年数据
        if current_year is not None and year != current_year:
            ds_year = xr.concat(results_for_year, dim="time")
            output_dir = os.path.join(output_root, str(current_year))
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f"{current_year}_weekly_samples.nc")
            ds_year.to_netcdf(output_path)
            print(f"{current_year} 年数据合并保存成功：{output_path}")

            # 清空上一年数据
            results_for_year = []

        current_year = year

        # 计算过去20周（第4周到第23周）和10周（第4周到第13周）的日期列表
        past_20_weeks = [target_date - pd.Timedelta(weeks=i) for i in range(4, 24)]
        past_10_weeks = [target_date - pd.Timedelta(weeks=i) for i in range(4, 14)]

        # 选择数据，自动忽略不存在的时间
        tas_hist_all = ds['tas'].sel(time=past_20_weeks)
        gh200_hist_all = ds['gh200'].sel(time=past_10_weeks)
        gh200_300_hist_all = ds['gh200_300'].sel(time=past_10_weeks)
        gh200_500_hist_all = ds['gh200_500'].sel(time=past_10_weeks)

        # 打印每年第一个target_date对应的原始时间
        if year not in year_check:
            print(f"{year} 年对应的第一个 target_date：{target_date}, tas_hist 对应时间：{tas_hist_all.time.values}")
            year_check[year] = True

        data_vars = {}
        coords = {"time": [target_date], "latitude": ds.latitude, "longitude": ds.longitude}

        # 历史20周 tas，变量名：tas_hist_0,...,tas_hist_19
        for i in range(tas_hist_all.time.size):
            data_vars[f"tas_hist_{i}"] = (("latitude", "longitude"), tas_hist_all.isel(time=i).values)

        # 历史10周 gh200
        for i in range(gh200_hist_all.time.size):
            data_vars[f"gh200_hist_{i}"] = (("latitude", "longitude"), gh200_hist_all.isel(time=i).values)

        # 历史10周 gh200_300
        for i in range(gh200_300_hist_all.time.size):
            data_vars[f"gh200_300_hist_{i}"] = (("latitude", "longitude"), gh200_300_hist_all.isel(time=i).values)

        # 历史10周 gh200_500
        for i in range(gh200_500_hist_all.time.size):
            data_vars[f"gh200_500_hist_{i}"] = (("latitude", "longitude"), gh200_500_hist_all.isel(time=i).values)

        ds_sample = xr.Dataset(data_vars=data_vars, coords=coords)

        results_for_year.append(ds_sample)

    except Exception as e:
        print(f"{target_date.date()} ❌ 错误：{e}")

# 循环结束后保存最后一年数据
if results_for_year:
    ds_year = xr.concat(results_for_year, dim="time")
    output_dir = os.path.join(output_root, str(current_year))
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{current_year}_weekly_samples.nc")
    ds_year.to_netcdf(output_path)
    print(f"{current_year} 年数据合并保存成功：{output_path}")

In [ ]:
# 8 合并周日数据
import os
import xarray as xr

# 路径设置
daily_dir = "/mnt/g/次季节模型/数据/日tas"
weekly_dir = "/mnt/g/次季节模型/数据/周数据"
merged_yearly_dir = "/mnt/g/次季节模型/数据/merged_yearly"
os.makedirs(merged_yearly_dir, exist_ok=True)

years = sorted([
    f.split('_')[-1].split('.')[0]
    for f in os.listdir(daily_dir)
    if f.startswith("tas_weekly_split_") and f.endswith(".nc")
])

def merge_one_year(year):
    daily_file = os.path.join(daily_dir, f"tas_weekly_split_{year}.nc")
    weekly_file = os.path.join(weekly_dir, f"{year}_weekly_samples.nc")
    merged_file = os.path.join(merged_yearly_dir, f"merged_{year}.nc")

    print(f"处理年份 {year} ...")
    ds_daily = xr.open_dataset(daily_file)
    ds_weekly = xr.open_dataset(weekly_file)

    # 简单检查时间维度是否有重叠或冲突，可根据需求添加
    # 合并两个 Dataset，变量不同，可以用 merge
    ds_merged = xr.merge([ds_daily, ds_weekly])

    # 保存年合并文件
    ds_merged.to_netcdf(merged_file)
    print(f"保存年合并文件: {merged_file}")

def merge_all_years(final_output):
    yearly_files = sorted([
        os.path.join(merged_yearly_dir, f) for f in os.listdir(merged_yearly_dir)
        if f.endswith(".nc")
    ])
    print(f"开始合并所有年文件，共{len(yearly_files)}个")

    ds_list = [xr.open_dataset(f) for f in yearly_files]
    # 按time维度合并所有年份数据，concat合并time维度
    ds_all = xr.concat(ds_list, dim="time")

    # 保存最终大文件
    ds_all.to_netcdf(final_output)
    print(f"最终合并文件保存至 {final_output}")

if __name__ == "__main__":
    for year in years:
        merge_one_year(year)
    final_output_file = "/mnt/g/次季节模型/7day.nc"
    merge_all_years(final_output_file)

In [ ]:
# 9 添加月预测数据
import os
import xarray as xr
import numpy as np
from tqdm import tqdm

# 输入目录和文件
merged_dir = "/mnt/g/次季节模型/数据/merged_yearly"
pred_file = "/mnt/g/次季节模型/数据/准备数据/predictions_1990_20251.nc"

# 输出目录（自动创建）
output_dir = "/mnt/g/次季节模型/数据/merged_with_pred"
os.makedirs(output_dir, exist_ok=True)

# 读取预测数据
ds_pred = xr.open_dataset(pred_file)
pred_t2m = ds_pred["pred_t2m"]

# 经度调整为 [0, 360]
pred_t2m = pred_t2m.assign_coords(longitude=(pred_t2m.longitude % 360)).sortby("longitude")

# 遍历每年数据
for year in range(2000, 2025):
    merged_path = os.path.join(merged_dir, f"merged_{year}.nc")
    if not os.path.exists(merged_path):
        print(f"⚠️ 文件不存在: {merged_path}")
        continue

    print(f"✅ 正在处理: {merged_path}")
    ds_merged = xr.open_dataset(merged_path)

    time_array = ds_merged["time"].values
    pred_month_values = []

    for t in time_array:
        t = np.datetime64(t)
        t_month = np.datetime64(f"{t.astype('datetime64[M]')}")
        if t_month not in pred_t2m.time:
            print(f"❌ 缺失月份预测数据: {t_month}")
            pred_month_values.append(np.full((len(ds_merged.latitude), len(ds_merged.longitude)), np.nan))
        else:
            pred_month_values.append(pred_t2m.sel(time=t_month).values)

    pred_t2m_month = xr.DataArray(
        data=np.stack(pred_month_values),
        dims=("time", "latitude", "longitude"),
        coords={
            "time": ds_merged.time,
            "latitude": ds_merged.latitude,
            "longitude": ds_merged.longitude
        },
        name="pred_t2m_month",
        attrs={"description": "每个time对应月份的月平均预测2m气温"}
    )

    ds_merged["pred_t2m_month"] = pred_t2m_month

    # 保存到新目录
    output_path = os.path.join(output_dir, f"merged_mon_{year}.nc")
    ds_merged.to_netcdf(output_path)
    print(f"💾 已保存到: {output_path}")


In [ ]:
# 10 添加地形数据
import os
import xarray as xr
import numpy as np

# 路径
input_dir = "/mnt/g/次季节模型/数据/merged_with_pred"
output_dir = "/mnt/g/次季节模型/数据/merged_with_pred_with_elev"
os.makedirs(output_dir, exist_ok=True)

# 地形数据路径
elev_file = "/mnt/g/次季节模型/数据/准备数据/etopo_1.5deg.nc"
ds_elev = xr.open_dataset(elev_file)

# 地形数据的经纬度变量名与范围
# ds_elev: lat (-90~90), lon (-180~180)
# 目标数据: latitude (-90~90), longitude (0~360)

# 将地形经度转换为 0~360 范围，方便对齐
def lon_180_to_360(lon):
    lon_360 = lon.copy()
    lon_360 = np.where(lon_360 < 0, lon_360 + 360, lon_360)
    return lon_360

ds_elev = ds_elev.assign_coords(lon=lon_180_to_360(ds_elev.lon))
# 按升序排序经度（确保与目标数据一致）
ds_elev = ds_elev.sortby('lon')

# 遍历输入目录的每个nc文件
for filename in sorted(os.listdir(input_dir)):
    if not filename.endswith(".nc"):
        continue
    input_path = os.path.join(input_dir, filename)
    print(f"处理文件: {input_path}")

    ds = xr.open_dataset(input_path)

    # 将地形数据重采样到目标文件的经纬度（latitude/longitude）坐标
    # 先重命名地形的纬度纬度名与目标一致方便插值
    ds_elev_renamed = ds_elev.rename({'lat': 'latitude', 'lon': 'longitude'})

    # 使用xarray的interp进行插值匹配坐标
    # 这里按纬度和经度双线性插值
    ds_elev_interp = ds_elev_renamed.interp(
        latitude=ds.latitude,
        longitude=ds.longitude,
        method="linear"
    )

    # ds_elev_interp的elevation是二维变量，无时间维度，扩展到时间维度与目标数据一致
    # 按照时间维度广播
    elev_expanded = ds_elev_interp['elevation'].expand_dims({'time': ds.time}, axis=0)
    elev_expanded = elev_expanded.transpose('time', 'latitude', 'longitude')

    # 新建一个DataArray保存广播后的地形数据
    elev_da = xr.DataArray(
        data=elev_expanded.values,
        dims=['time', 'latitude', 'longitude'],
        coords={'time': ds.time, 'latitude': ds.latitude, 'longitude': ds.longitude},
        attrs={
            'long_name': 'surface_elevation',
            'units': 'meters',
            'description': 'Surface elevation interpolated from etopo_1.5deg.nc and expanded along time dimension'
        }
    )

    # 添加新变量到原数据集
    ds = ds.assign(elevation=elev_da)

    # 保存新文件到输出目录
    output_path = os.path.join(output_dir, filename)
    print(f"保存新文件: {output_path}")
    ds.to_netcdf(output_path)
    ds.close()

print("全部处理完成！")


In [ ]:
# 11 合并数据
import xarray as xr
import os

# 输入目录和输出文件路径
input_dir = "/mnt/g/次季节模型/数据/merged_with_pred_with_elev"
output_file = "/mnt/g/次季节模型/数据/D.nc"

# 需要合并的年份列表
years = range(2020, 2025)

# 构造要打开的文件路径列表
file_paths = [os.path.join(input_dir, f"merged_mon_{year}.nc") for year in years]

# 使用 xarray.open_mfdataset 进行多文件合并，按 time 维度拼接
ds_merged = xr.open_mfdataset(file_paths, combine='by_coords', parallel=True)

# 保存合并后的数据集
ds_merged.to_netcdf(output_file)

print(f"合并完成，保存文件：{output_file}")


In [ ]:
# （2）地形
import xarray as xr
import numpy as np
import os

# 输入和输出路径
input_path = "/mnt/g/数据/ETOP高程数据/ETOPO_2022_v1_60s_N90W180_geoid.nc"
output_path = "/mnt/g/次季节模型/数据/准备数据/etopo_1.5deg.nc"

# 打开原始 ETOPO 数据
ds = xr.open_dataset(input_path)
z = ds["z"]

# 去除 NaN（保留非空值参与插值）
z_clean = z.where(np.isfinite(z), drop=True)

# 创建目标 1.5° 网格
target_lat = np.arange(-90, 90 + 1.5, 1.5)
target_lon = np.arange(-180, 180, 1.5)

# 使用 xarray 自带的插值功能
z_interp = z_clean.interp(
    lat=target_lat,
    lon=target_lon,
    method="linear"
    #method="nearest"
)

# 创建新数据集
ds_interp = xr.Dataset(
    {
        "elevation": (["lat", "lon"], z_interp.data)
    },
    coords={
        "lat": target_lat,
        "lon": target_lon
    },
    attrs={
        "title": "ETOPO interpolated to 1.5-degree grid",
        "source": "Interpolated from ETOPO 2022 60s dataset",
        "units": "meters"
    }
)

# 保存为 NetCDF 文件
os.makedirs(os.path.dirname(output_path), exist_ok=True)
ds_interp.to_netcdf(output_path)

print(f"✅ 成功保存差值结果到: {output_path}")
